# ¿Cómo se comportó nuestro sistema de salud bajo presión?"

## Mapa de Roles (El Índice de Asimetría):

### 1.1. Mapa de Roles: "Sumideros" y "Distribuidores" en la Macro-Red
[Introducción teórica breve]

Para entender cómo fluyeron los pacientes en el sistema, clasificamos a los hospitales según su comportamiento logístico utilizando el Índice de Asimetría. Este índice va de -1 a +1:

- Cercano a -1 (Distribuidores): Funcionan como "peajes" o centros de triaje. Reciben pacientes y los derivan rápidamente a mayor complejidad.
- Cercano a +1 (Sumideros): Funcionan como "embudos" o destinos finales. Reciben pacientes graves de toda la red y los retienen hasta el fin del caso.

In [1]:
# 12. TRAYECTORIAS: IMPACTO DE LA CONDICIÓN DE ENTRADA Y COMORBILIDADES
# =============================================================================

# 1. Rescatamos las variables clínicas desde el historial crudo (df_base)
# Para el riesgo, tomamos el del primer ingreso ('first')
# Para el respirador, buscamos si en ALGUN momento de toda su historia requirió ARM
df_clinico_extra = df_base.groupby('paciente_id').agg({
    'riesgo_clinico': 'first',
    'requiere_arm': lambda x: 1 if any(str(v).lower() in ['si', 'sí', 'true', '1'] for v in x) else 0
}).reset_index()

# Renombramos para que coincida con tu lógica
df_clinico_extra = df_clinico_extra.rename(columns={'requiere_arm': 'necesito_respirador'})

# 2. Se las pegamos a nuestro DataFrame de análisis (que ya tiene los saltos del paso 11)
df_pacientes_analisis = df_pacientes_analisis.merge(df_clinico_extra, on='paciente_id', how='left')

# 3. Analizamos por Riesgo Clínico Inicial
riesgo_trayectoria = df_pacientes_analisis.groupby('riesgo_clinico').agg(
    total=('paciente_id', 'count'),
    promedio_saltos_red=('cantidad_saltos', 'mean'),
    tasa_requiere_arm_=('necesito_respirador', 'mean') # mean() de 0s y 1s da la proporción exacta
)

# Pasamos a porcentaje
riesgo_trayectoria['tasa_requiere_arm_%'] = riesgo_trayectoria['tasa_requiere_arm_'] * 100

print("➤ COMPLEJIDAD DE LA TRAYECTORIA SEGÚN EL RIESGO CLÍNICO AL INGRESAR:")
display(riesgo_trayectoria[['total', 'promedio_saltos_red', 'tasa_requiere_arm_%']].sort_values('tasa_requiere_arm_%', ascending=False))

NameError: name 'df_base' is not defined

### ⚠️ OPCIÓN D (Clúster 3): Filtrado Manual de Trayectorias
*Nota: Copiado repetitivo de df_pacientes_trayectorias para análisis individual.*

In [ ]:
# 14. ANÁLISIS DEL PACIENTE: EVOLUCIÓN DE MORTALIDAD SEGÚN TRAYECTORIA (Corregido)
# =============================================================================

# 1. Partimos de nuestra tabla maestra de pacientes (1 fila por persona)
df_pacientes_analisis = df_pacientes_trayectorias.copy()

# Asignar cada paciente al período correspondiente según su fecha de ingreso a la red
def asignar_periodo(fecha):
    if pd.isna(fecha): return 'Sin Dato'
    for titulo, inicio, fin in PERIODOS:
        if pd.to_datetime(inicio) <= fecha <= pd.to_datetime(fin):
            return titulo
    return 'Fuera de Rango'

df_pacientes_analisis['periodo_ingreso'] = df_pacientes_analisis['fecha_ingreso_red'].apply(asignar_periodo)

# 2. Traemos la cantidad de saltos EN AMBULANCIA
# (Chequeo por si no corriste la celda anterior recién)
if 'df_red_real' not in locals():
    df_red_real = df_aristas_traslados[df_aristas_traslados['es_ambulancia'] == 'ambulancia'].copy()

saltos_por_paciente = df_red_real.groupby('paciente_id').size().reset_index(name='cantidad_saltos')
df_pacientes_analisis = df_pacientes_analisis.merge(saltos_por_paciente, on='paciente_id', how='left')
df_pacientes_analisis['cantidad_saltos'] = df_pacientes_analisis['cantidad_saltos'].fillna(0).astype(int)

# 3. Agrupamos saltos largos
df_pacientes_analisis['categoria_saltos'] = df_pacientes_analisis['cantidad_saltos'].apply(
    lambda x: '0 Traslados' if x == 0 else ('1 Traslado' if x == 1 else '2 o más Traslados')
)

# Filtramos los datos válidos y agrupamos
df_validos = df_pacientes_analisis[df_pacientes_analisis['periodo_ingreso'].isin([p[0] for p in PERIODOS])]

mortalidad_periodo = df_validos.groupby(['periodo_ingreso', 'categoria_saltos']).agg(
    total_pacientes=('paciente_id', 'count'),
    muertes=('motivo_fin_caso', lambda x: (x == 'muerte').sum())
).reset_index()

mortalidad_periodo['tasa_mortalidad_%'] = (mortalidad_periodo['muertes'] / mortalidad_periodo['total_pacientes']) * 100

# Ordenar los períodos cronológicamente
orden_periodos = [p[0] for p in PERIODOS]
mortalidad_periodo['periodo_ingreso'] = pd.Categorical(mortalidad_periodo['periodo_ingreso'], categories=orden_periodos, ordered=True)
mortalidad_periodo = mortalidad_periodo.sort_values('periodo_ingreso')

# ---------------------------------------------------------
# MOSTRAMOS LA TABLA PARA CONTROL DE VOLUMEN (N)
# ---------------------------------------------------------
print("➤ DETALLE DE MORTALIDAD Y VOLUMEN DE PACIENTES (N) POR PERÍODO:")
# Mostramos solo las filas donde hubo al menos 1 paciente
display(mortalidad_periodo[mortalidad_periodo['total_pacientes'] > 0])
print("\n" + "="*80 + "\n")

# ---------------------------------------------------------
# GRAFICAMOS
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 7))

# Filtramos donde haya más de 6 pacientes para que el % no sea engañoso (ej: 1 muerto de 1 paciente = 100%)
sns.barplot(data=mortalidad_periodo[mortalidad_periodo['total_pacientes'] > 6], 
            x='periodo_ingreso', y='tasa_mortalidad_%', hue='categoria_saltos', 
            palette=['#a1dab4', '#41b6c4', '#225ea8'], ax=ax)

ax.set_title("Evolución de la Mortalidad según Cantidad de Traslados por Período", fontsize=16, fontweight='bold')
ax.set_xlabel("Período de la Pandemia")
ax.set_ylabel("Tasa de Mortalidad (%)")
ax.legend(title="Trayectoria del Paciente")

# Agregamos los porcentajes arriba de las barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)

plt.tight_layout()
guardar_pdf('des_barras_mortalidad_tipo_trayectoria', subcarpeta='desenlaces')
plt.show()

In [ ]:
# 14.A INVESTIGACIÓN DE ANOMALÍAS: Los sobrevivientes con múltiples traslados
# =============================================================================
# Filtramos exactamente ese grupo
casos_raros = df_pacientes_analisis[
    (df_pacientes_analisis['periodo_ingreso'] == 'Intermedia') &
    (df_pacientes_analisis['categoria_saltos'] == '2 o más Traslados')
].copy()

print(f"➤ Analizando {len(casos_raros)} pacientes anómalos...")

# 1. Rescatamos la "historia clínica" de df_base SOLO para estos pacientes raros
ids_raros = casos_raros['paciente_id']
info_clinica_extra = df_base[df_base['paciente_id'].isin(ids_raros)].groupby('paciente_id').agg(
    edad=('edad', 'first'),
    estado_ingreso=('estado_ingreso', 'first'),
    estado_ultimo=('estado_ultimo', 'last'),
    ultimo_hospital=('hospital_origen', 'last') # Buscamos la última cama que pisaron
).reset_index()

# 2. Le pegamos esa info a nuestra tablita de casos raros
casos_raros = casos_raros.merge(info_clinica_extra, on='paciente_id', how='left')

# 3. Definimos las columnas clave (usamos motivo_fin_caso que es la correcta ahora)
cols_interes = ['paciente_id', 'edad', 'estado_ingreso', 'estado_ultimo', 
                'motivo_fin_caso', 'cantidad_saltos', 'ultimo_hospital']

display(casos_raros[cols_interes])

# 4. Veamos también a qué hospitales fueron destinados finalmente
print("\n➤ Últimos hospitales donde estuvieron registrados:")
resumen_hospitales = casos_raros.groupby('ultimo_hospital').size().reset_index(name='cantidad_pacientes')
display(resumen_hospitales.sort_values('cantidad_pacientes', ascending=False))

In [ ]:
# 14.B GRÁFICO DE EVOLUCIÓN (VERSIÓN LÍNEAS / POINTPLOT)
# =============================================================================
fig, ax = plt.subplots(figsize=(14, 7))

# Usamos pointplot en lugar de barplot
# ACÁ ESTABA EL ERROR: Cambiamos 'total' por 'total_pacientes'
sns.pointplot(data=mortalidad_periodo[mortalidad_periodo['total_pacientes'] > 0], 
              x='periodo_ingreso', 
              y='tasa_mortalidad_%', 
              hue='categoria_saltos', 
              palette=['#2ca02c', '#ff7f0e', '#d62728'], # Verde, Naranja, Rojo
              markers=['o', 's', 'D'], # Círculo, Cuadrado, Diamante
              linestyles=['-', '--', '-.'],
              linewidth=2.5,
              ax=ax)

ax.set_title("Evolución de la Mortalidad según Cantidad de Traslados por Período", fontsize=16, fontweight='bold')
ax.set_xlabel("Período de la Pandemia", fontsize=12)
ax.set_ylabel("Tasa de Mortalidad (%)", fontsize=12)
ax.legend(title="Trayectoria del Paciente", fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

# Opcional: Anotar la anomalía de N=14 directamente en el gráfico
ax.annotate('Posible subregistro\n(N=14, 0 muertes)', 
            xy=(1, 0), # Coordenada (Indice 1=Intermedia, Y=0%)
            xytext=(1.2, 5), # Posición del texto
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
            fontsize=10, bbox=dict(boxstyle="round", alpha=0.1))

plt.tight_layout()
guardar_pdf('evo_lineas_mortalidad_tipo_trayectoria_por_periodo', subcarpeta='evolucion')
plt.show()

In [ ]:
# 15. ANÁLISIS DEL PACIENTE: EL FENÓMENO DEL "PING-PONG" INTRA-PREDIO
# =============================================================================

# 0. Asegurarnos de que existe la columna es_ambulancia (por seguridad)
if 'es_ambulancia' not in df_aristas_traslados.columns:
    df_aristas_traslados['es_ambulancia'] = df_aristas_traslados.apply(requiere_ambulancia, axis=1)

# 1. Filtramos SOLAMENTE los traslados que NO fueron en ambulancia (Intra-predio / Camilla)
# USAMOS EL NOMBRE NUEVO: df_aristas_traslados
df_camilla = df_aristas_traslados[df_aristas_traslados['es_ambulancia'] == False].copy()

# 2. Contamos cuántos rebotes internos tuvo cada paciente
rebotes_internos = df_camilla.groupby('paciente_id').size().reset_index(name='cantidad_rebotes_internos')

# 3. Lo unimos a nuestro DF maestro de pacientes (USAMOS df_pacientes_trayectorias)
df_pacientes_pingpong = df_pacientes_trayectorias.merge(rebotes_internos, on='paciente_id', how='left')
df_pacientes_pingpong['cantidad_rebotes_internos'] = df_pacientes_pingpong['cantidad_rebotes_internos'].fillna(0).astype(int)

# 4. Creamos la categoría de análisis
def categorizar_rebote(cant):
    if cant == 0:
        return 'Sin rebotes internos'
    elif cant == 1:
        return '1 Traslado a Módulo (Normal)'
    else:
        return '2 o más (Ping-Pong)'

df_pacientes_pingpong['tipo_pingpong'] = df_pacientes_pingpong['cantidad_rebotes_internos'].apply(categorizar_rebote)

# 5. Calculamos la mortalidad agrupada por este nuevo fenómeno
mortalidad_pingpong = df_pacientes_pingpong.groupby('tipo_pingpong').agg(
    total_pacientes=('paciente_id', 'count'),
    muertes=('motivo_fin_caso', lambda x: (x == 'muerte').sum())
).reset_index()

mortalidad_pingpong['tasa_mortalidad_%'] = (mortalidad_pingpong['muertes'] / mortalidad_pingpong['total_pacientes']) * 100

# Ordenamos para visualizar mejor
mortalidad_pingpong = mortalidad_pingpong.sort_values('tasa_mortalidad_%')

print("➤ IMPACTO DEL REBOTE INTRA-PREDIO (CAMILLA) EN LA MORTALIDAD GLOBAL:")
display(mortalidad_pingpong)

# Graficamos
fig, ax = plt.subplots(figsize=(10, 6))
# Usamos un degradado de colores para enfatizar el aumento de riesgo
sns.barplot(data=mortalidad_pingpong, x='tipo_pingpong', y='tasa_mortalidad_%', 
            palette='YlOrRd', ax=ax)

ax.set_title("Efecto del 'Ping-Pong' Intra-Predio en la Mortalidad del Paciente", fontsize=14, fontweight='bold')
ax.set_xlabel("Dinámica Interna (Mismo Predio)")
ax.set_ylabel("Tasa de Mortalidad (%)")

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=12, fontweight='bold')

plt.tight_layout()
guardar_pdf('des_barras_mortalidad_pingpong_global', subcarpeta='general')
plt.show()

In [ ]:
# 16. DESARMANDO EL MISTERIO: Anatomía del paciente "Ping-Pong" (Sin usar ARM)
# =============================================================================

# 1. Rescatamos los datos clínicos desde la historia en bruto (df_base)
# Buscamos si pasó por críticas en algún momento y su riesgo clínico inicial
df_clinico_pingpong = df_base.groupby('paciente_id').agg({
    'paso_criticas': lambda x: 'Si' if any(str(v).lower() in ['si', 'sí', 'true'] for v in x) else 'No',
    'riesgo_clinico': 'first'
}).reset_index()

# Cruzamos estos datos clínicos con nuestra tabla de análisis
df_pacientes_pingpong = df_pacientes_pingpong.merge(df_clinico_pingpong, on='paciente_id', how='left')

# (Nota: Omitimos recalcular 'dias_estadia_total' porque ya existe en este DataFrame 
# gracias a nuestra nueva arquitectura relacional).

# 2. Analizamos el paso por Terapia Intensiva (Críticas) y Riesgo Clínico
columnas_analisis = ['paso_criticas', 'riesgo_clinico']

print("➤ PERFIL CLÍNICO DEL PACIENTE SEGÚN SU DINÁMICA DE TRASLADOS:")
for col in columnas_analisis:
    print(f"\n--- Distribución de '{col.upper()}' ---")
    cruce = pd.crosstab(
        df_pacientes_pingpong['tipo_pingpong'], 
        df_pacientes_pingpong[col], 
        normalize='index' # Porcentaje por fila
    ) * 100
    display(cruce.round(1).style.format("{:.1f}%"))

# 3. Analizamos el Tiempo y Destino
print("\n➤ RESULTADO LOGÍSTICO (Días de internación y Destino final):")
metricas_logisticas = df_pacientes_pingpong.groupby('tipo_pingpong').agg(
    total_pacientes=('paciente_id', 'count'),
    promedio_dias_estadia=('dias_estadia_total', 'mean'),
    mediana_dias_estadia=('dias_estadia_total', 'median'),
    # ACÁ CORREGIDO: Buscamos 'alta' porque unificamos los nombres antes
    alta_domiciliaria_pct=('motivo_fin_caso', lambda x: (x == 'alta').mean() * 100)
).round(1)

display(metricas_logisticas)

# 4. Gráfico para hacerlo visual en tu presentación
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_pacientes_pingpong[df_pacientes_pingpong['dias_estadia_total'] < 60], # Cortamos outliers de más de 2 meses para el dibujo
            x='tipo_pingpong', y='dias_estadia_total', 
            palette=['#cccccc', '#ffc107', '#dc3545'], ax=ax)

ax.set_title("Días de Internación según la Dinámica Intra-Predio", fontsize=14, fontweight='bold')
ax.set_xlabel("Dinámica Interna")
ax.set_ylabel("Días de Estadía Totales")
plt.tight_layout()
plt.show()

➤ TOP HOSPITALES 'SUMIDERO' (Retienen pacientes - Índice cercano a +1):
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>hospital</th>
      <th>recibidos</th>
      <th>enviados</th>
      <th>volumen_total</th>
      <th>indice_asimetria</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>6</th>
      <td>Módulo Hospitalario 11 - FV</td>
      <td>275.0</td>
      <td>31.0</td>
      <td>306.0</td>
      <td>0.797386</td>
    </tr>
    <tr>
      <th>0</th>
      <td>El Cruce</td>
      <td>103.0</td>
      <td>21.0</td>
      <td>124.0</td>
      <td>0.661290</td>
    </tr>
    <tr>
      <th>7</th>
      <td>Módulo Hospitalario 9 - AB</td>
      <td>103.0</td>
      <td>32.0</td>
      <td>135.0</td>
      <td>0.525926</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Iriarte</td>
      <td>18.0</td>
      <td>17.0</td>
      <td>35.0</td>
      <td>0.028571</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Lucio Meléndez</td>
      <td>102.0</td>
      <td>116.0</td>
      <td>218.0</td>
      <td>-0.064220</td>
    </tr>
  </tbody>
</table>
</div>

➤ TOP HOSPITALES 'DISTRIBUIDORES' (Derivan rápido - Índice cercano a -1):
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>hospital</th>
      <th>recibidos</th>
      <th>enviados</th>
      <th>volumen_total</th>
      <th>indice_asimetria</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>13</th>
      <td>UPA 5 - AB</td>
      <td>28.0</td>
      <td>81.0</td>
      <td>109.0</td>
      <td>-0.486239</td>
    </tr>
    <tr>
      <th>11</th>
      <td>UPA 11 - FV</td>
      <td>13.0</td>
      <td>39.0</td>
      <td>52.0</td>
      <td>-0.500000</td>
    </tr>
    <tr>
      <th>5</th>
      <td>Módulo Hospitalario 10 - QU</td>
      <td>11.0</td>
      <td>36.0</td>
      <td>47.0</td>
      <td>-0.531915</td>
    </tr>
    <tr>
      <th>9</th>
      <td>Oñativia</td>
      <td>5.0</td>
      <td>30.0</td>
      <td>35.0</td>
      <td>-0.714286</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Mi Pueblo</td>
      <td>39.0</td>
      <td>293.0</td>
      <td>332.0</td>
      <td>-0.765060</td>
    </tr>
  </tbody>
</table>
</div>

[Tus Conclusiones / Hallazgos]

El destino final: Como era de esperarse, los hospitales "Modulo Hospitalario 11" y "El Cruce" (ambos en Florencio Varela) actuaron como sumideros, absorbiendo la carga de pacientes críticos.

La primera línea: Los centros "Mi Pueblo" (en Florencio Varela) y "Onativia" (en Almirante Brown)fueron los grandes distribuidores, demostrando su rol clave conteniendo la demanda inicial y derivando a los que realmente lo necesitaban.

matriz logistica

### 1.2. Evolución del Rol durante la Pandemia
[Insertar Gráfico de Líneas]
(Acá pegás el gráfico de evolución temporal con las zonas Azul, Negra y Roja)

[Tus Conclusiones / Hallazgos]

Flexibilidad de la red: Acá tenés que mirar tu gráfico y contar lo que ves. ¿Algún hospital empezó en el medio (equilibrio) y de repente en la Ola 2 se fue para arriba (se saturó y se volvió sumidero)? ¿Los distribuidores mantuvieron su rol siempre igual? Esto demuestra si la red fue estática o si mutó por el estrés del COVID-19.

1.3. La Micro-Red: El Fenómeno del "Ping-Pong" Intra-predio
[Introducción]

Al aislar los traslados en ambulancia (Macro-red), descubrimos una segunda capa de saturación: la Micro-red (traslados en camilla dentro del mismo hospital o hacia módulos anexos). A este fenómeno de alta rotación interna lo llamamos "Ping-Pong".

[Insertar Gráfico Doble (Boxplot + Líneas)]
(Pegás la imagen 1x2 que te generó la última celda de tu código)

[Tus Conclusiones / Hallazgos]

La Paradoja de Supervivencia: Los datos muestran que los pacientes con múltiples rebotes internos (Ping-Pong) curiosamente tienen menor mortalidad, a pesar de tener un riesgo clínico inicial alto.

El Paciente Crónico: El gráfico de estadía (boxplot) explica esta paradoja: el "Ping-Pong" no mata al paciente al instante, sino que genera estadías ultra-prolongadas. Son pacientes graves que logran estabilizarse pero quedan "atrapados" en el sistema rebotando entre camas de menor y mayor complejidad según la disponibilidad del día.